# SQL practice 9: arrays, structs, maps and JSON

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-practice-09-arrays-structs-and-json.ipynb)

14 exercises on arrays, explode, higher-order functions, structs, maps, JSON and VARIANT. Each exercise names the [reference entries](sql-reference-09-arrays-structs-and-json.ipynb) it practises, shows the exact expected output, and hides a reference solution you can reveal when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0 (Spark SQL).*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local Spark session, the shared tables as temporary views: `customers`, `orders`, `order_items`, `products`, `events` and `employees` (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry, two pairs of employees tie on salary, Dana out-earns her manager and Jo has no department), and the helpers from [SQL reference 9](sql-reference-09-arrays-structs-and-json.ipynb#setup-and-sample-data). Exercises that need other data create it in their input cell. This notebook also defines `posts` and `raw_events`.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# Shared sample tables as temporary views. Each statement is ordinary Spark SQL.
TABLES = {
    "customers": """CREATE OR REPLACE TEMP VIEW customers AS SELECT * FROM VALUES
  (1, 'Acme Corp',      'North', DATE'2024-01-03'),
  (2, 'Bright Retail',  'North', DATE'2024-02-11'),
  (3, 'Cyan Traders',   'South', DATE'2024-03-15'),
  (4, 'Delta Grocers',  NULL,    DATE'2024-05-02'),   -- region never arrived from the CRM
  (5, 'Echo Supplies',  'West',  DATE'2024-06-20'),
  (6, 'Foxtrot Foods',  'West',  DATE'2024-08-01'),
  (7, 'Gamma Sports',   'East',  DATE'2024-09-14'),   -- 7 and 8 have never ordered
  (8, 'Hotel Homeware', 'East',  DATE'2024-11-05')
AS customers(customer_id, name, region, signup_date)""",
    "orders": """CREATE OR REPLACE TEMP VIEW orders AS SELECT order_id, customer_id, status, order_ts, CAST(amount AS DECIMAL(12,2)) AS amount FROM VALUES
  (1,  1, 'paid',     TIMESTAMP'2026-01-05 09:15:00', 120.00),
  (2,  1, 'shipped',  TIMESTAMP'2026-01-05 10:05:00', 84.50),
  (3,  1, 'returned', TIMESTAMP'2026-01-05 11:40:00', 30.00),
  (4,  2, 'paid',     TIMESTAMP'2026-01-06 15:20:00', 260.00),
  (5,  2, 'placed',   TIMESTAMP'2026-01-07 08:05:00', 55.25),
  (6,  3, 'shipped',  TIMESTAMP'2026-01-07 17:45:00', 149.00),
  (7,  3, 'paid',     TIMESTAMP'2026-01-09 12:10:00', 310.00),
  (8,  4, 'returned', TIMESTAMP'2026-01-10 19:30:00', 89.90),
  (9,  4, 'shipped',  TIMESTAMP'2026-01-12 08:55:00', 1180.00),
  (10, 5, 'paid',     TIMESTAMP'2026-01-12 13:00:00', 74.00),
  (11, 6, 'placed',   TIMESTAMP'2026-01-13 10:25:00', 210.00),
  (12, 6, 'shipped',  TIMESTAMP'2026-01-14 00:30:00', 96.40)   -- just after midnight UTC
AS orders(order_id, customer_id, status, order_ts, amount)""",
    "order_items": """CREATE OR REPLACE TEMP VIEW order_items AS SELECT order_id, sku, qty, CAST(unit_price AS DECIMAL(8,2)) AS unit_price FROM VALUES
  (1, 'SKU-4001', 2, 12.50), (1, 'SKU-4007', 1, 95.00), (2, 'SKU-4002', 1, 34.50), (2, 'SKU-4009', 2, 25.00),
  (3, 'SKU-4002', 1, 30.00), (4, 'SKU-4003', 1, 149.00), (4, 'SKU-4004', 2, 55.50), (5, 'SKU-4005', 1, 55.25),
  (6, 'SKU-4003', 1, 149.00), (7, 'SKU-4006', 1, 189.00), (7, 'SKU-4008', 1, 121.00), (8, 'SKU-4010', 1, 89.90),
  (9, 'SKU-4011', 20, 59.00), (10, 'SKU-4012', 2, 37.00), (11, 'SKU-4013', 1, 210.00),
  (12, 'SKU-4014', 1, 41.40), (12, 'SKU-4005', 1, 55.00)
AS order_items(order_id, sku, qty, unit_price)""",
    "products": """CREATE OR REPLACE TEMP VIEW products AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
  ('SKU-4001', 'Kitchen', 12.50), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4003', 'Garden', 149.00),
  ('SKU-4004', 'Garden', 55.50), ('SKU-4005', 'Pantry', 55.25), ('SKU-4006', 'Outdoor', 189.00),
  ('SKU-4007', 'Kitchen', 95.00), ('SKU-4008', 'Outdoor', 121.00), ('SKU-4009', 'Pantry', 25.00),
  ('SKU-4010', 'Garden', 89.90), ('SKU-4011', 'Pantry', 59.00), ('SKU-4012', 'Kitchen', 37.00),
  ('SKU-4013', 'Outdoor', 210.00)   -- SKU-4014 has no product row
AS products(sku, category, list_price)""",
    "events": """CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM VALUES
  (1, 'view', TIMESTAMP'2026-01-06 09:00:00'), (1, 'add_to_cart', TIMESTAMP'2026-01-06 09:02:00'),
  (1, 'purchase', TIMESTAMP'2026-01-06 09:10:00'), (2, 'view', TIMESTAMP'2026-01-07 10:00:00'),
  (2, 'purchase', TIMESTAMP'2026-01-07 10:40:00'), (3, 'view', TIMESTAMP'2026-01-06 11:15:00'),
  (3, 'view', TIMESTAMP'2026-01-08 11:20:00'), (4, 'add_to_cart', TIMESTAMP'2026-01-09 20:05:00'),
  (4, 'purchase', TIMESTAMP'2026-01-10 08:30:00'), (5, 'view', TIMESTAMP'2026-01-11 14:45:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00'), (5, 'purchase', TIMESTAMP'2026-01-12 12:30:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00')   -- an SDK retry committed this event twice
AS events(user_id, event, ts)""",
    "employees": """CREATE OR REPLACE TEMP VIEW employees AS SELECT * FROM VALUES
  (1,  'Ada',    'Engineering', CAST(NULL AS INT), 185000, DATE'2018-03-01'),
  (2,  'Brett',  'Engineering', 1,    150000, DATE'2019-07-15'),
  (3,  'Chen',   'Engineering', 1,    150000, DATE'2020-01-06'),
  (4,  'Dana',   'Engineering', 2,    155000, DATE'2022-09-12'),   -- out-earns her manager
  (5,  'Eli',    'Sales',       1,    130000, DATE'2019-02-01'),
  (6,  'Farah',  'Sales',       5,    95000,  DATE'2021-05-17'),
  (7,  'Gus',    'Sales',       5,    95000,  DATE'2023-01-09'),
  (8,  'Hana',   'Finance',     1,    140000, DATE'2017-11-20'),
  (9,  'Ivan',   'Finance',     8,    88000,  DATE'2024-04-01'),
  (10, 'Jo',     NULL,          8,    70000,  DATE'2025-06-30')   -- department not assigned yet
AS employees(emp_id, name, dept, manager_id, salary, hire_date)""",
}
for statement in TABLES.values():
    spark.sql(statement)

import contextlib
import io
import logging
import re


def show(query, n=50, **args):
    """Run a query and print up to n rows without truncating values. Keyword arguments bind :name parameters."""
    spark.sql(query, args=args or None).show(n, truncate=False)


def plan(query, mode="formatted"):
    """The plan EXPLAIN prints for a query, as text."""
    return spark.sql(f"EXPLAIN {mode.upper()} {query}").first()[0]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:120])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

# Semi-structured sample data for this notebook.
spark.sql("""
CREATE OR REPLACE TEMP VIEW posts AS SELECT * FROM VALUES
  (1, 'Spring planting guide', array('garden', 'howto', 'seasonal')),
  (2, 'Knife care',            array('kitchen', 'howto')),
  (3, 'New arrivals',          array()),
  (4, 'BBQ season',            array('outdoor', 'seasonal', 'garden'))
AS posts(post_id, title, tags)
""")
_ = spark.sql("""
CREATE OR REPLACE TEMP VIEW raw_events AS SELECT * FROM VALUES
  (1, '{"type": "checkout", "user": {"id": 7, "tier": "gold"}, "items": [{"sku": "SKU-4001", "qty": 2, "price": 12.5}, {"sku": "SKU-4007", "qty": 1, "price": 95.0}]}'),
  (2, '{"type": "checkout", "user": {"id": 9}, "items": [{"sku": "SKU-4003", "qty": 1, "price": 149.0}]}'),
  (3, '{"type": "page_view", "user": {"id": 7, "tier": "gold"}, "page": "/garden"}'),
  (4, '{"type": "checkout", "user": {"id": 11, "tier": "silver"}, "items": []}')
AS raw_events(event_id, payload)
""")

## Exercises

9 easy, 4 medium, 1 hard. Write your query in the empty cell under each exercise, run it with `show(q)` or `spark.sql(q).show()`, and compare it with the expected output (columns, values and row order).

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Posts per tag](#exercise-1-posts-per-tag) | Easy | [`explode` and friends](sql-reference-09-arrays-structs-and-json.ipynb#explode-and-friends) |
| 2 | [Posts without tags still listed](#exercise-2-posts-without-tags-still-listed) | Easy | [`explode` and friends](sql-reference-09-arrays-structs-and-json.ipynb#explode-and-friends) |
| 3 | [First and last tag](#exercise-3-first-and-last-tag) | Easy | [Building and reading arrays](sql-reference-09-arrays-structs-and-json.ipynb#building-and-reading-arrays) |
| 4 | [Seasonal how-to posts](#exercise-4-seasonal-how-to-posts) | Easy | [Building and reading arrays](sql-reference-09-arrays-structs-and-json.ipynb#building-and-reading-arrays) |
| 5 | [SKUs per order as a sorted list](#exercise-5-skus-per-order-as-a-sorted-list) | Easy | [`explode` and friends](sql-reference-09-arrays-structs-and-json.ipynb#explode-and-friends) |
| 6 | [Largest order per customer with a struct](#exercise-6-largest-order-per-customer-with-a-struct) | Medium | [Structs](sql-reference-09-arrays-structs-and-json.ipynb#structs) |
| 7 | [Discounted prices with transform](#exercise-7-discounted-prices-with-transform) | Easy | [Higher-order functions](sql-reference-09-arrays-structs-and-json.ipynb#higher-order-functions) |
| 8 | [Basket totals with aggregate](#exercise-8-basket-totals-with-aggregate) | Medium | [Higher-order functions](sql-reference-09-arrays-structs-and-json.ipynb#higher-order-functions) |
| 9 | [Product attributes from a map](#exercise-9-product-attributes-from-a-map) | Easy | [Maps](sql-reference-09-arrays-structs-and-json.ipynb#maps) |
| 10 | [Fields from JSON payloads](#exercise-10-fields-from-json-payloads) | Easy | [Extracting from JSON strings](sql-reference-09-arrays-structs-and-json.ipynb#extracting-from-json-strings) |
| 11 | [Flatten JSON line items](#exercise-11-flatten-json-line-items) | Medium | [Parsing with `from_json` and writing with `to_json`](sql-reference-09-arrays-structs-and-json.ipynb#parsing-with-from_json-and-writing-with-to_json), [`explode` and friends](sql-reference-09-arrays-structs-and-json.ipynb#explode-and-friends) |
| 12 | [Build a JSON document per customer](#exercise-12-build-a-json-document-per-customer) | Medium | [Parsing with `from_json` and writing with `to_json`](sql-reference-09-arrays-structs-and-json.ipynb#parsing-with-from_json-and-writing-with-to_json), [Structs](sql-reference-09-arrays-structs-and-json.ipynb#structs) |
| 13 | [Typed fields from VARIANT](#exercise-13-typed-fields-from-variant) | Easy | [`VARIANT` (Spark 4.0)](sql-reference-09-arrays-structs-and-json.ipynb#variant-spark-40) |
| 14 | [Tag co-occurrence](#exercise-14-tag-co-occurrence) | Hard | [`explode` and friends](sql-reference-09-arrays-structs-and-json.ipynb#explode-and-friends), [Set and sort operations on arrays](sql-reference-09-arrays-structs-and-json.ipynb#set-and-sort-operations-on-arrays) |

## Exercise 1: Posts per tag

**Difficulty:** Easy · about 5 min · **Practises:** [`explode` and friends](sql-reference-09-arrays-structs-and-json.ipynb#explode-and-friends)

*Exploding arrays to count their elements.*

Return `tag` and `posts` (number of posts with that tag), most common first, then alphabetically.

**Expected output**

| tag | posts |
| --- | --- |
| garden | 2 |
| howto | 2 |
| seasonal | 2 |
| kitchen | 1 |
| outdoor | 1 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT tag, count(*) AS posts
    FROM posts
    LATERAL VIEW explode(tags) t AS tag
    GROUP BY tag
    ORDER BY posts DESC, tag
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 2: Posts without tags still listed

**Difficulty:** Easy · about 5 min · **Practises:** [`explode` and friends](sql-reference-09-arrays-structs-and-json.ipynb#explode-and-friends)

*explode drops rows with empty arrays; explode_outer keeps them.*

Return `post_id` and `tag` with one row per tag, and one row with a NULL tag for posts without tags, ordered by `post_id`, `tag`.

**Expected output**

| post_id | tag |
| --- | --- |
| 1 | garden |
| 1 | howto |
| 1 | seasonal |
| 2 | howto |
| 2 | kitchen |
| 3 | null |
| 4 | garden |
| 4 | outdoor |
| 4 | seasonal |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT post_id, tag
    FROM posts
    LATERAL VIEW OUTER explode(tags) t AS tag
    ORDER BY post_id, tag
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 3: First and last tag

**Difficulty:** Easy · about 3 min · **Practises:** [Building and reading arrays](sql-reference-09-arrays-structs-and-json.ipynb#building-and-reading-arrays)

*Index bases differ between get/[] and element_at, and ANSI mode errors on out-of-range indexes.*

Return `post_id`, `first_tag`, `last_tag` and `n_tags`, ordered by `post_id`. Posts without tags have NULLs.

**Expected output**

| post_id | first_tag | last_tag | n_tags |
| --- | --- | --- | --- |
| 1 | garden | seasonal | 3 |
| 2 | kitchen | howto | 2 |
| 3 | null | null | 0 |
| 4 | outdoor | garden | 3 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT post_id, get(tags, 0) AS first_tag, try_element_at(tags, -1) AS last_tag, size(tags) AS n_tags
    FROM posts
    ORDER BY post_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 4: Seasonal how-to posts

**Difficulty:** Easy · about 3 min · **Practises:** [Building and reading arrays](sql-reference-09-arrays-structs-and-json.ipynb#building-and-reading-arrays)

*Filtering on array membership.*

Return the `title` of posts tagged both `seasonal` and either `garden` or `outdoor`, ordered by `title`.

**Expected output**

| title |
| --- |
| BBQ season |
| Spring planting guide |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT title
    FROM posts
    WHERE array_contains(tags, 'seasonal') AND arrays_overlap(tags, array('garden', 'outdoor'))
    ORDER BY title
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 5: SKUs per order as a sorted list

**Difficulty:** Easy · about 5 min · **Practises:** [`explode` and friends](sql-reference-09-arrays-structs-and-json.ipynb#explode-and-friends)

*Collapsing rows back into arrays, deterministically.*

Return `order_id`, `skus` (a comma-separated string of the order's SKUs in alphabetical order) and `lines`, for orders 1 to 4, ordered by `order_id`.

**Expected output**

| order_id | skus | lines |
| --- | --- | --- |
| 1 | SKU-4001,SKU-4007 | 2 |
| 2 | SKU-4002,SKU-4009 | 2 |
| 3 | SKU-4002 | 1 |
| 4 | SKU-4003,SKU-4004 | 2 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT order_id, array_join(array_sort(collect_list(sku)), ',') AS skus, count(*) AS lines
    FROM order_items
    WHERE order_id <= 4
    GROUP BY order_id
    ORDER BY order_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 6: Largest order per customer with a struct

**Difficulty:** Medium · about 5 min · **Practises:** [Structs](sql-reference-09-arrays-structs-and-json.ipynb#structs)

*The struct arg-max trick returns a whole row in one aggregate.*

Return `customer_id`, `order_id`, `amount` and `status` of each customer's largest order, using `max(struct(...))`, ordered by `customer_id`.

**Expected output**

| customer_id | order_id | amount | status |
| --- | --- | --- | --- |
| 1 | 1 | 120.00 | paid |
| 2 | 4 | 260.00 | paid |
| 3 | 7 | 310.00 | paid |
| 4 | 9 | 1180.00 | shipped |
| 5 | 10 | 74.00 | paid |
| 6 | 11 | 210.00 | placed |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT customer_id, top.order_id, top.amount, top.status
    FROM (
        SELECT customer_id, max(struct(amount, order_id, status)) AS top
        FROM orders
        GROUP BY customer_id
    )
    ORDER BY customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 7: Discounted prices with transform

**Difficulty:** Easy · about 5 min · **Practises:** [Higher-order functions](sql-reference-09-arrays-structs-and-json.ipynb#higher-order-functions)

*Array math without exploding.*

Using `baskets` below, return `basket_id`, `discounted` (each price × 0.9, rounded to 2, as an array) and `over_50` (how many discounted prices are above 50), ordered by `basket_id`.

In [ ]:
spark.sql("CREATE OR REPLACE TEMP VIEW baskets AS SELECT * FROM VALUES (1, array(10D, 60D, 100D)), (2, array(55D)), (3, CAST(array() AS ARRAY<DOUBLE>)) AS b(basket_id, prices)")

**Expected output**

| basket_id | discounted | over_50 |
| --- | --- | --- |
| 1 | [9.0, 54.0, 90.0] | 2 |
| 2 | [49.5] | 0 |
| 3 | [] | 0 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT basket_id,
           transform(prices, p -> round(p * 0.9, 2)) AS discounted,
           size(filter(prices, p -> p * 0.9 > 50)) AS over_50
    FROM baskets
    ORDER BY basket_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 8: Basket totals with aggregate

**Difficulty:** Medium · about 10 min · **Practises:** [Higher-order functions](sql-reference-09-arrays-structs-and-json.ipynb#higher-order-functions)

*Folding arrays of structs into a single number.*

Using `raw_events`, parse checkout payloads with `from_json` and return `event_id`, `user_id`, `units` (sum of qty) and `basket_value` (sum of qty × price), ordered by `event_id`. Checkouts with no items have 0 for both.

**Expected output**

| event_id | user_id | units | basket_value |
| --- | --- | --- | --- |
| 1 | 7 | 3 | 120.0 |
| 2 | 9 | 1 | 149.0 |
| 4 | 11 | 0 | 0.0 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    WITH parsed AS (
        SELECT event_id, from_json(payload, 'type STRING, user STRUCT<id: INT, tier: STRING>, items ARRAY<STRUCT<sku: STRING, qty: INT, price: DOUBLE>>') AS e
        FROM raw_events
    )
    SELECT event_id, e.user.id AS user_id,
           aggregate(e.items, 0, (acc, i) -> acc + i.qty) AS units,
           aggregate(e.items, 0D, (acc, i) -> acc + i.qty * i.price) AS basket_value
    FROM parsed
    WHERE e.type = 'checkout'
    ORDER BY event_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 9: Product attributes from a map

**Difficulty:** Easy · about 5 min · **Practises:** [Maps](sql-reference-09-arrays-structs-and-json.ipynb#maps)

*Sparse attributes stored as maps.*

Using `product_attrs` below, return `sku`, `color` (`n/a` when missing), `size` and `n_attrs`, ordered by `sku`.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW product_attrs AS SELECT * FROM VALUES
  ('SKU-4001', map('color', 'steel', 'size', 'M')), ('SKU-4003', map('size', 'XL')), ('SKU-4006', map('color', 'green', 'size', 'L', 'finish', 'matte'))
AS p(sku, attrs)
""")

**Expected output**

| sku | color | size | n_attrs |
| --- | --- | --- | --- |
| SKU-4001 | steel | M | 2 |
| SKU-4003 | n/a | XL | 1 |
| SKU-4006 | green | L | 3 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT sku, coalesce(attrs['color'], 'n/a') AS color, attrs['size'] AS size, size(attrs) AS n_attrs
    FROM product_attrs
    ORDER BY sku
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 10: Fields from JSON payloads

**Difficulty:** Easy · about 5 min · **Practises:** [Extracting from JSON strings](sql-reference-09-arrays-structs-and-json.ipynb#extracting-from-json-strings)

*Path extraction for a quick look at raw events.*

From `raw_events`, return `event_id`, `type`, `user_id` (as INT) and `tier` (`none` when missing), ordered by `event_id`, using `get_json_object`.

**Expected output**

| event_id | type | user_id | tier |
| --- | --- | --- | --- |
| 1 | checkout | 7 | gold |
| 2 | checkout | 9 | none |
| 3 | page_view | 7 | gold |
| 4 | checkout | 11 | silver |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT event_id,
           get_json_object(payload, '$.type') AS type,
           CAST(get_json_object(payload, '$.user.id') AS INT) AS user_id,
           coalesce(get_json_object(payload, '$.user.tier'), 'none') AS tier
    FROM raw_events
    ORDER BY event_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 11: Flatten JSON line items

**Difficulty:** Medium · about 10 min · **Practises:** [Parsing with `from_json` and writing with `to_json`](sql-reference-09-arrays-structs-and-json.ipynb#parsing-with-from_json-and-writing-with-to_json), [`explode` and friends](sql-reference-09-arrays-structs-and-json.ipynb#explode-and-friends)

*Nested event payloads become one row per line item for a fact table.*

Parse `raw_events` with `from_json`, explode the items of checkout events, and return `event_id`, `user_id`, `line` (1-based position), `sku`, `qty` and `line_total` (qty × price), ordered by `event_id`, `line`.

**Expected output**

| event_id | user_id | line | sku | qty | line_total |
| --- | --- | --- | --- | --- | --- |
| 1 | 7 | 1 | SKU-4001 | 2 | 25.0 |
| 1 | 7 | 2 | SKU-4007 | 1 | 95.0 |
| 2 | 9 | 1 | SKU-4003 | 1 | 149.0 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    WITH parsed AS (
        SELECT event_id, from_json(payload, 'type STRING, user STRUCT<id: INT, tier: STRING>, items ARRAY<STRUCT<sku: STRING, qty: INT, price: DOUBLE>>') AS e
        FROM raw_events
    )
    SELECT event_id, e.user.id AS user_id, pos + 1 AS line, item.sku, item.qty, item.qty * item.price AS line_total
    FROM parsed
    LATERAL VIEW posexplode(e.items) t AS pos, item
    WHERE e.type = 'checkout'
    ORDER BY event_id, line
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 12: Build a JSON document per customer

**Difficulty:** Medium · about 10 min · **Practises:** [Parsing with `from_json` and writing with `to_json`](sql-reference-09-arrays-structs-and-json.ipynb#parsing-with-from_json-and-writing-with-to_json), [Structs](sql-reference-09-arrays-structs-and-json.ipynb#structs)

*APIs and document stores want nested JSON built from relational rows.*

For customers 1 to 3, return `customer_id` and `doc`: a JSON object with `name`, `region` and `orders` (an array of `{"id", "amount"}` objects sorted by id), ordered by `customer_id`.

**Expected output**

| customer_id | doc |
| --- | --- |
| 1 | {"name":"Acme Corp","region":"North","orders":[{"id":1,"amount":120.00},{"id":2,"amount":84.50},{"id":3,"amount":30.00}]} |
| 2 | {"name":"Bright Retail","region":"North","orders":[{"id":4,"amount":260.00},{"id":5,"amount":55.25}]} |
| 3 | {"name":"Cyan Traders","region":"South","orders":[{"id":6,"amount":149.00},{"id":7,"amount":310.00}]} |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT c.customer_id,
           to_json(named_struct(
               'name', c.name,
               'region', c.region,
               'orders', array_sort(collect_list(named_struct('id', o.order_id, 'amount', o.amount)))
           )) AS doc
    FROM customers c
    JOIN orders o ON o.customer_id = c.customer_id
    WHERE c.customer_id <= 3
    GROUP BY c.customer_id, c.name, c.region
    ORDER BY c.customer_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 13: Typed fields from VARIANT

**Difficulty:** Easy · about 5 min · **Practises:** [`VARIANT` (Spark 4.0)](sql-reference-09-arrays-structs-and-json.ipynb#variant-spark-40)

*VARIANT parses once and extracts typed values safely.*

Parse `raw_events.payload` with `parse_json` and return `event_id`, `type`, `user_id` (INT), `page` (NULL when absent) and `first_qty` (INT quantity of the first item, NULL when absent), ordered by `event_id`.

**Expected output**

| event_id | type | user_id | page | first_qty |
| --- | --- | --- | --- | --- |
| 1 | checkout | 7 | null | 2 |
| 2 | checkout | 9 | null | 1 |
| 3 | page_view | 7 | /garden | null |
| 4 | checkout | 11 | null | null |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    SELECT event_id,
           variant_get(v, '$.type', 'string') AS type,
           variant_get(v, '$.user.id', 'int') AS user_id,
           try_variant_get(v, '$.page', 'string') AS page,
           try_variant_get(v, '$.items[0].qty', 'int') AS first_qty
    FROM (SELECT event_id, parse_json(payload) AS v FROM raw_events)
    ORDER BY event_id
"""
result = spark.sql(q)
result.show(truncate=False)

## Exercise 14: Tag co-occurrence

**Difficulty:** Hard · about 15 min · **Practises:** [`explode` and friends](sql-reference-09-arrays-structs-and-json.ipynb#explode-and-friends), [Set and sort operations on arrays](sql-reference-09-arrays-structs-and-json.ipynb#set-and-sort-operations-on-arrays)

*Pairs of labels that appear together, the array version of market-basket analysis.*

Return `tag_a`, `tag_b` (with `tag_a < tag_b`) and `posts` (number of posts carrying both tags), ordered by `posts` descending, then `tag_a`, `tag_b`.

**Expected output**

| tag_a | tag_b | posts |
| --- | --- | --- |
| garden | seasonal | 2 |
| garden | howto | 1 |
| garden | outdoor | 1 |
| howto | kitchen | 1 |
| howto | seasonal | 1 |
| outdoor | seasonal | 1 |

In [ ]:
q = """
-- your query here
"""
# show(q)

In [ ]:
q = """
    WITH pairs AS (
        SELECT post_id, a AS tag_a, b AS tag_b
        FROM posts
        LATERAL VIEW explode(array_distinct(tags)) ta AS a
        LATERAL VIEW explode(array_distinct(tags)) tb AS b
        WHERE a < b
    )
    SELECT tag_a, tag_b, count(DISTINCT post_id) AS posts
    FROM pairs
    GROUP BY tag_a, tag_b
    ORDER BY posts DESC, tag_a, tag_b
"""
result = spark.sql(q)
result.show(truncate=False)

**Next:** look anything up in [SQL reference 9: arrays, structs, maps and JSON](sql-reference-09-arrays-structs-and-json.ipynb), or try mixed interview problems in [SQL interview drills](sql-13-interview-drills.ipynb).